In [0]:
%run ./adls_auth

In [0]:
# Databricks notebook source
# ==============================================================================
# Pipeline Audit Logging Utility (log_pipeline_run)
# ==============================================================================

# 1. Define Input Widgets
dbutils.widgets.text("p_run_id", "", "ADF Run ID")
dbutils.widgets.text("p_pipeline_name", "master", "Pipeline Name")
dbutils.widgets.text("p_activity_name", "start", "Activity Name")
dbutils.widgets.text("p_status", "STARTED", "Status")
dbutils.widgets.text("p_error_message", "", "Error Message")

# 2. Extract Parameter Values
v_run_id = dbutils.widgets.get("p_run_id").strip()
v_pipeline_name = dbutils.widgets.get("p_pipeline_name").strip()
v_activity_name = dbutils.widgets.get("p_activity_name").strip()
v_status = dbutils.widgets.get("p_status").strip().upper()

v_error_raw = dbutils.widgets.get("p_error_message").strip()
if not v_error_raw or v_error_raw.upper() in ["NONE", "NULL", ""]:
    v_error_message = None
else:
    v_error_message = v_error_raw
if not v_run_id:
    raise ValueError("p_run_id widget parameter cannot be empty.")

# 3. Path Configuration
table_path = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/pipeline_run_log"

# 4. Ensure Control Table Exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{table_path}` (
        run_id STRING,
        pipeline_name STRING,
        activity_name STRING,
        status STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        duration_seconds LONG,
        error_message STRING
    )
    USING DELTA
""")

# 5. Execute Audit Logging Logic
if v_status == "STARTED":
    spark.sql(f"""
        INSERT INTO delta.`{table_path}` (
            run_id, pipeline_name, activity_name, status, start_time, end_time, duration_seconds, error_message
        )
        VALUES (
            '{v_run_id}', '{v_pipeline_name}', '{v_activity_name}', 'STARTED', current_timestamp(), NULL, 0, NULL
        )
    """)
    print(f"Logged STARTED status for run_id: {v_run_id}")

else:
    clean_error_msg = v_error_message.replace("'", "''") if v_error_message else None
    error_sql_val = f"'{clean_error_msg}'" if clean_error_msg else "NULL"

    spark.sql(f"""
        MERGE INTO delta.`{table_path}` t
        USING (
            SELECT 
                '{v_run_id}' AS run_id,
                current_timestamp() AS curr_time
        ) s
        ON t.run_id = s.run_id
        WHEN MATCHED THEN UPDATE SET
            t.status = '{v_status}',
            t.activity_name = '{v_activity_name}',
            t.end_time = s.curr_time,
            t.duration_seconds = unix_timestamp(s.curr_time) - unix_timestamp(t.start_time),
            t.error_message = {error_sql_val}
        WHEN NOT MATCHED THEN INSERT (
            run_id, pipeline_name, activity_name, status, start_time, end_time, duration_seconds, error_message
        ) VALUES (
            '{v_run_id}', '{v_pipeline_name}', '{v_activity_name}', '{v_status}', current_timestamp(), current_timestamp(), 0, {error_sql_val}
        )
    """)
    print(f"Updated {v_status} status for run_id: {v_run_id}")